# <span style="color:red"> Lecture 22: NumPy, Pandas, and Vectorization </span>

<font size = "4">

**Overview:** In this notebook we will

- Measure how long code takes to run with the `timeit` library
- See why Python `for` loops are slow, and see how "list comprehensions" can make a (modest) improvement.

- Make big improvements by replacing loops with **vectorized** operations in NumPy and pandas.

- Vectorize `if`/`elif`/`else` logic with Boolean masks, `np.where`, and `np.select`

In [2]:
import numpy as np
import pandas as pd

# <span style="color:red"> I. Timing Code with timeit </span>

<font size = "4">

- Not only should code **work**, but it should be **fast**. People want code that will execute in seconds, not minutes or hours.

- Before we can make code faster, we need a reliable way to measure it. We can use Python's built-in `timeit` library is one way to do this.

- The `timeit.default_timer` function returns the current time in seconds. How do we measure "current" time in seconds? Seconds since January 1st, 2026? Or some other reference point?

- According to Python documentation, "The reference point of the returned value is undefined, so that only the **difference** between the results of two calls is valid"

In [3]:
from timeit import default_timer

current_time = default_timer()
print(f"The previous line of code executed {current_time} seconds after an undefined reference point")
print()

start = default_timer()
print("Hello World!")
x = 5
y = 2 * 1_000_000
z = y/x
print("Goodbye World!")
stop = default_timer()

print(f"The previous 5 lines of code took {stop - start} seconds to execute")
print(f"The previous 5 lines of code took {(stop - start)*1e3} milliseconds to execute")
print(f"The previous 5 lines of code took {(stop - start)*1e6} microseconds to execute")
print(f"The previous 5 lines of code took {(stop - start)*1e9} nanoseconds to execute")

The previous line of code executed 653981.9402205 seconds after an undefined reference point

Hello World!
Goodbye World!
The previous 5 lines of code took 0.0006264001131057739 seconds to execute
The previous 5 lines of code took 0.6264001131057739 milliseconds to execute
The previous 5 lines of code took 626.4001131057739 microseconds to execute
The previous 5 lines of code took 626400.1131057739 nanoseconds to execute


<font size = "4">

An easy way to use `default_timer` is to gather the instructions inside of a function:

In [4]:
def loop_sum(numbers):
    """Add up a list of numbers with a for loop."""
    total = 0
    for x in numbers:
        total = total + x
    return total

numbers = list(range(1_000_000))    # Integers between 0 (inclusive) and 1,000,000 (exclusive)

start = default_timer()
result = loop_sum(numbers)
stop = default_timer()

print("Computed sum:", result)
print("Time (seconds):", stop - start)

Computed sum: 499999500000
Time (seconds): 0.2376819000346586


<font size = "4">


Run the cell above a few times. The time changes every run, because your computer is doing other things at the same time (your internet browser, background updates, etc.) **A single measurement is noisy.**

We can get a better estimate by **averaging** over multiple runs:

In [5]:
numbers = list(range(1_000_000))

n_runs = 10
time_list = []
for k in range(n_runs):
    start = default_timer()
    result = loop_sum(numbers)
    stop = default_timer()
    time_list.append(stop - start)


print("Total time for", n_runs, "runs:", sum(time_list), "seconds")
print("Average time per run:", sum(time_list) / n_runs, "seconds")

Total time for 10 runs: 1.5620381000917405 seconds
Average time per run: 0.15620381000917405 seconds


<font size = "4">

Actually, `timeit.timeit` function can do this averaging for us. 

The easiest way to use it is to wrap up your code inside a function with **no arguments**

In [6]:
from timeit import timeit

numbers = list(range(1_000_000))

def run_loop_sum():
    return loop_sum(numbers)

run_loop_sum()

499999500000

In [7]:
from timeit import timeit

numbers = list(range(1_000_000))

def run_loop_sum():
    return loop_sum(numbers)

n_runs = 10
total_time = timeit(run_loop_sum, number = n_runs)

print("Total time for", n_runs, "runs:", total_time, "seconds")
print("Average time per run:", total_time / n_runs, "seconds")


KeyboardInterrupt: 

<font size = "4">

For a simple case like this, you can use a `lambda` function (with no arguments) instead of defining a function using `def`.



In [ ]:
run_loop_sum = lambda: loop_sum(numbers)

n_runs = 10
total_time = timeit(run_loop_sum, number = n_runs)

In [ ]:
n_runs = 10
total_time = timeit(lambda: loop_sum(numbers), number = n_runs)
print("Total time for", n_runs, "runs:", total_time, "seconds")
print("Average time per run:", total_time / n_runs, "seconds")

Total time for 10 runs: 2.0711433000396937 seconds
Average time per run: 0.20711433000396937 seconds


<font size = "4">

**Exercise #1:**

Of course, we didn't need to write `loop_sum`, because Python's built-in `sum` function already exists. In fact, it is **faster** than our function.

- Continue to use `numbers = list(range(1_000_000))`

- Use `timeit` with 10 runs (number = 10) to find the **average time per run** of the command `sum(numbers)`.

- Use `timeit` with 10 runs to find the average time per run of the command `loop_sum(numbers)` (i.e. copy and paste the code from above).

- Print both average times to the screen. Then compute and print the value of the **speed-up:**

$$\textrm{speed-up} = \frac{\textrm{slower time}}{\textrm{faster time}}$$

In [ ]:
# your code here
n_runs = 10



sum_time = timeit(lambda: sum(numbers), number=n_runs)
loop_time = timeit(lambda: loop_sum(numbers), number = n_runs)

print("Avg. time for sum: ", sum_time / n_runs )
print("Avg. time for loop:", loop_time / n_runs )
print("Speed-up:          ", loop_time /  sum_time )


Avg. time for sum:  0.02544654000084847
Avg. time for loop: 0.1989803500007838
Speed-up:           7.819544425063255


In [ ]:
# your code here
n_runs = 10

method1 = lambda: sum(numbers)
def method2():
    return loop_sum(numbers)


sum_time = timeit(method1, number=n_runs)
loop_time = timeit(method2, number = n_runs)

print("Avg. time for sum: ", sum_time / n_runs )
print("Avg. time for loop:", loop_time / n_runs )
print("Speed-up:          ", loop_time /  sum_time )

Avg. time for sum:  0.018214100005570798
Avg. time for loop: 0.16477935999864712
Speed-up:           9.046802199847876


# <span style="color:red"> II. Rewriting loops with list comprehension </span>

<font size = "4">

- The built-in `sum` function and our `loop_sum` do the same additions, so why is the built-in one so much faster? Python must "interpret" each line of code and convert it into what is called "machine code". This process must be done **every** pass through the loop.

- Using `sum` means this interpretation step only executes once.

- If we avoid loops, we can get faster code.

- But loops aren't always bad. They are easy to read, and fast enough for small sets of data. But if you're looping over **thousands or millions** of values, you might want to look for a faster alternative.

**List comprehensions** are one mechanism for avoiding these multiple interpretation steps. A list comprehension builds a list in one line:
```python
[expression for item in iterable]
[expression for item in iterable if condition]
```

We'll walk through the first variant using code that squares every number in a list. See if you can figure the other one out in exercise #2.



In [ ]:
# Method #1: use a loop

def squares_loop(numbers):
    result = []
    for x in numbers:
        result.append(x ** 2)
    return result

# Method #2: use list comprehension
def squares_comprehension(numbers):
    return [x ** 2 for x in numbers]

def make_ones(numbers):
    return [1 for x in numbers]

print(squares_loop([1, 2, 3, 4]))
print(squares_comprehension([1, 2, 3, 4]))
print(make_ones([1, 2, 3, 4]))

[1, 4, 9, 16]
[1, 4, 9, 16]
[1, 1, 1, 1]


In [ ]:
[x**2 for x in [1, 2, 3, 4]]

[1, 4, 9, 16]

<font size = "4">

Now let's compare the running time:

In [ ]:
numbers = list(range(1_000_000))

n_runs = 10
loop_avg = timeit(lambda: squares_loop(numbers), number = n_runs) / n_runs
comp_avg = timeit(lambda: squares_comprehension(numbers), number = n_runs) / n_runs

print("for loop:          ", loop_avg)
print("list comprehension:", comp_avg)
print("speedup:           ", loop_avg / comp_avg)

for loop:           0.38084351000143213
list comprehension: 0.24537863000296056
speedup:            1.5520647009759454


<font size = "4">

It's slightly faster, but not much to write home about. To get a **big** speed-up we need to use vectorization with NumPy.

<font size = "4">

**Exercise #2:** Below, the list `temps_f` contains one million temperatures in degrees Fahrenheit. The function `warm_celsius_loop` converts every temperature **above freezing** (above 32 °F) to Celsius, and skips the rest.

- Try to rewrite the function `warm_celsius_loop` to use list comprehension. The pattern for how to do so is as follows:
```python
[expression for item in iterable if condition]
```
- Compare the running time with the original loop-based function.

In [ ]:
import random

# Loop-based code
random.seed(160)
temps_f = [round(random.uniform(-10, 100), 1) for i in range(1_000_000)]


def warm_celsius_loop(temps):
    result = []
    for t in temps:
        if t > 32:
            result.append((t - 32) * 5 / 9)
    return result


print(temps_f[:5])
print(warm_celsius_loop(temps_f[:5]))

[3.5, 27.8, 9.2, 54.7, 68.2]
[12.611111111111112, 20.11111111111111]


In [ ]:

def warm_celsius_comp(temps):
    return [(t - 32) * 5 / 9 for t in temps if t > 32]

method1 = lambda: warm_celsius_comp(temps_f)
method2 = lambda: warm_celsius_loop(temps_f)

n_runs = 5
comp_time = timeit(method1, number=n_runs)
loop_time = timeit(method2, number = n_runs)

print("Avg. time for comp: ", comp_time / n_runs )
print("Avg. time for loop: ", loop_time / n_runs )
print("Speed-up:           ", loop_time /  comp_time )

Avg. time for comp:  0.4241113000083715
Avg. time for loop:  0.39933684000279757
Speed-up:            0.9415850037358474


# <span style="color:red"> III. Vectorization with NumPy </span>

<font size = "4">


**What is vectorization?**

- "Not writing Python `for` loops"
- Operating on a whole array at once, instead of one element at a time
- NumPy pushes the loop down into fast, compiled **C** code
- This allows for huge speed increases

Here are some key differences between a list and a NumPy array

| Python list | NumPy array |
| --- | --- |
| Can mix types: `[1, "a", 2.5]` | Every element has the **same type** (its `dtype`) |
| Stores references to objects scattered in memory | Stores the raw numbers **side by side** in memory |
| Loops run in Python | Loops run in C |

Because every element has the same type, NumPy can skip part of the interpretation step (checking types) for every element, and run one tight loop in C.

<font size = "4">

We already saw how we can use NumPy to avoid loops. 

- Arithmetic operations on arrays are carried out element-by-element, with no loops.

- We can also do something like add a single number to a NumPy array; the operation is applied to every element. This is called **broadcasting**

In [ ]:
small_array = np.array([1, 2, 3, 4])

print(small_array ** 2)
print(small_array + 10)   # broadcasting
print(small_array * small_array)
print(np.sqrt(small_array))

[ 1  4  9 16]
[11 12 13 14]
[ 1  4  9 16]
[1.         1.41421356 1.73205081 2.        ]


<font size = "4">

How do element-wise operations with NumPy arrays compare with loops and list comprehension?

In [ ]:
numbers = list(range(1_000_000))
arr = np.arange(1_000_000)


n_runs = 10

numpy_avg = timeit(lambda: arr**2, number = n_runs) / n_runs
loop_avg = timeit(lambda: squares_loop(numbers), number = n_runs) / n_runs
comp_avg = timeit(lambda: [x**2 for x in numbers], number = n_runs) / n_runs

print("for loop avg:          ", loop_avg, "seconds")
print("list comprehension avg:", comp_avg, "seconds")
print("NumPy avg:             ", numpy_avg, "seconds")
print("Numpy vs loop speed-up:", loop_avg / numpy_avg)
print("Numpy vs comp speed-up:", comp_avg / numpy_avg)

for loop avg:           0.42826141000259665 seconds
list comprehension avg: 0.32775646999944 seconds
NumPy avg:              0.007084189995657653 seconds
Numpy vs loop speed-up: 60.4531231185365
Numpy vs comp speed-up: 46.2659062222135


<font size = "4">

**One catch:** converting a list into an array also takes time, because NumPy has to visit every element of the list once.

Vectorization pays off when the data is **already** contained in an array (or a DataFrame). Load your data once, keep it in NumPy or pandas, and don't convert back and forth between lists.

In [ ]:
n_runs = 10
time_convert = timeit(lambda: np.array(numbers), number = n_runs) / n_runs
print("Converting a list to an array:", time_convert, "seconds")

Converting a list to an array: 0.11725083999335766 seconds


### <span style="color:red"> Aggregations and universal functions </span>

<font size = "4">


- NumPy arrays have a variety of methods that allow you to skip a loop: `.sum()`, `.mean()`, `.min()`, `.max()`, `.std()`

- NumPy functions like `np.sqrt`, `np.exp`, `np.log`, `np.abs`, and `np.round` work on every element at once. These are called **universal functions** ("ufuncs").

In [ ]:
n_runs = 10
time_numpy_sum = timeit(lambda: arr.sum(), number = n_runs) / n_runs
time_loop_sum = timeit(lambda: loop_sum(numbers), number = n_runs) / n_runs
time_sum_sum = timeit(lambda: sum(numbers), number = n_runs) / n_runs

print("Average time in seconds")
print("loop_sum:  ", time_loop_sum)
print("sum:       ", time_sum_sum)
print("arr.sum(): ", time_numpy_sum)

print()
print("Numpy vs. loop speedup:", time_loop_sum / time_numpy_sum)
print("Numpy vs. sum speedup:", time_sum_sum / time_numpy_sum)

Average time in seconds
loop_sum:   0.17155935000628234
sum:        0.015137269999831915
arr.sum():  0.001732389989774674

Numpy vs. loop speedup: 99.03044407950918
Numpy vs. sum speedup: 8.737795813401558


In [ ]:
print(f"np.sqrt is a {type(np.sqrt)}")

np.sqrt is a <class 'numpy.ufunc'>


### <span style="color:red"> Boolean arrays </span>

<font size = "4">

Comparing an array to a value (or another array) returns an array of `True`/`False` values, called a **Boolean mask**. Masks let us:

- **Filter**: `arr[mask]` keeps only the elements where the mask is `True`

- **Count**: `True` counts as 1 and `False` as 0, so `mask.sum()` counts the `True` values and `mask.mean()` gives the fraction that are `True`

In [ ]:
values = np.array([3, 8, 1, 9, 4, 7])
mask = values > 5
print(mask)
print(values[5])
print(values[mask])
print()
print("Number of elements larger than 5:          ", mask.sum())
print("Proportion/fraction that are larger than 5:", mask.mean())

[False  True False  True False  True]
7
[8 9 7]

Number of elements larger than 5:           3
Proportion/fraction that are larger than 5: 0.5


<font size = "4">

To use Boolean operations with masks, use the following **element by element** operators:

- `&` instead of `and`

- `|` instead of `or`

- `~` instead of `not`

**Always wrap each individual comparison in parentheses!** We'll see why in Section V.

In [ ]:
expr1 = (values > 2) & (values < 8)     # "and"
expr2 = (values < 2) | (values > 8)     # "or"
print("Array:                        ", values)
print("Between 2 and 8 (exclusive):  ", expr1)
print("Less than 2 or greater than 8:", expr2)
print()
print("Mask:      ", mask)
print("'Not' mask:", ~mask)

Array:                         [3 8 1 9 4 7]
Between 2 and 8 (exclusive):   [ True False False False  True  True]
Less than 2 or greater than 8: [False False  True  True False False]

Mask:       [False  True False  True False  True]
'Not' mask: [ True False  True False  True False]


<font size = "4">

**Exercise # 3 (estimating π):** If we drop random points into a 1-by-1 square, the fraction that lands inside the quarter circle $x^2 + y^2 \le 1$ is about $\pi / 4$. So

$$\pi \approx 4 \times (\text{fraction of points inside the circle})$$

The cell below creates one million random points (`x` and `y` are NumPy arrays) and a loop-based function, `estimate_pi_loop`.

- Write `estimate_pi_vec(x, y)` that returns the same estimate **without a loop**. Hint: build a Boolean mask, then use `.mean()` or `.sum()`.

- Time both functions on `x` and `y`. Use the argument `number = 5` for `timeit`. Store the average time per run in `time_pi_loop` and `time_pi_vec`, and print the speedup. 


In [ ]:
# Loop based code
np.random.seed(160)
n_points = 1_000_000

# Generate points between 0 and 1
x = np.random.uniform(size=n_points)
y = np.random.uniform(size=n_points)

def estimate_pi_loop(x, y):
    inside = 0
    for i in range(len(x)):
        if x[i] ** 2 + y[i] ** 2 <= 1:
            inside = inside + 1
    return 4 * inside / len(x)

pi_est = estimate_pi_loop(x, y)

print("Estimate for pi is:", pi_est)

Estimate for pi is: 3.142952


In [ ]:
# Your code here

def estimate_pi_vec(x, y):
    return 4 * (x ** 2 + y ** 2 <= 1).mean()

n_runs = 10
time_loop = timeit(lambda: estimate_pi_loop(x, y), number = n_runs) / n_runs
time_vec = timeit(lambda: estimate_pi_vec(x, y), number = n_runs) / n_runs


print("Average time in seconds")
print("loop est:  ", time_loop)
print("vec est:   ", time_vec)

print()
print("Numpy vs. loop speedup:", time_loop / time_vec)


Average time in seconds
loop est:   1.8336260700016282
vec est:    0.03348461000714451

Numpy vs. loop speedup: 54.76026358408812


<font size = "4">

Notice that `estimate_pi_loop` is even slower than the loops in Section II. Looping over a **NumPy array** one element at a time is the worst of both worlds: every `x[i]` has to create a new Python number object. If your data is in an array, use array operations.

### <span style="color:red"> Multidimensional arrays </span>

<font size = "4">


Real data often comes as a **table** of numbers: one row per student and one column per assignment, one row per day and one column per sensor, and so on. A NumPy array can have two (or more) **dimensions**. Accessing elements of multi-dimensional NumPy arrays is similar to pandas DataFrames with `iloc`

- `.shape` gives the size of each dimension: `(rows, columns)`
- `arr[i, j]` is the value in row "i", column "j"
- `arr[i]` is all of row "i", and `arr[:, j]` is all of column "j"

In [ ]:
tiny = np.array([ [1,  2,  3,  4],
                  [5,  6,  7,  8],
                  [9, 10, 11, 12] ])

tiny = np.array([ [12,  2,  3,  9],
                  [8,  10,  7,  5],
                  [4, 6, 11, 1] ])

display(tiny)

print(tiny.shape)
print(tiny[1, 2])
print(tiny[1, :])
print(tiny[:, 2])

array([[12,  2,  3,  9],
       [ 8, 10,  7,  5],
       [ 4,  6, 11,  1]])

(3, 4)
7
[ 8 10  7  5]
[ 3  7 11]


### <span style="color:red"> Summing over an axis </span>

<font size = "4">

`tiny.sum()` adds up **every** value. To get one total per column, or one per row, use the `axis` argument:

- `axis = 0` collapses the **rows**, leaving one result per **column**: shape `(3, 4)` becomes `(4,)`
- `axis = 1` collapses the **columns**, leaving one result per **row**: shape `(3, 4)` becomes `(3,)`

A good way to remember it: the axis you name is the one that **disappears** from the shape.
```text
             axis = 1  →
          [[ 1,  2,  3,  4],     → 10
 axis = 0  [ 5,  6,  7,  8],     → 26
    ↓      [ 9, 10, 11, 12]]     → 42
            ↓   ↓   ↓   ↓
           15  18  21  24
```

`.mean()`, `.max()`, `.min()`, `.std()`, and `.argmax()` (the **position** of the largest value) all accept `axis` too. pandas uses the same idea: `df.sum(axis = 0)` gives one total per column, and `df.apply(func, axis = 1)` calls `func` once per row.

In [ ]:
display(tiny)

print("Sum over all elements:          ", tiny.sum())
print("Sum over the rows:              ", tiny.sum(axis = 0))
print("Sum over the columns:           ",tiny.sum(axis = 1))
print("Max over the rows:              ", tiny.max(axis = 0))
print("Max *position* over the columns:", tiny.argmax(axis = 1))

array([[12,  2,  3,  9],
       [ 8, 10,  7,  5],
       [ 4,  6, 11,  1]])

Sum over all elements:           78
Sum over the rows:               [24 18 21 15]
Sum over the columns:            [26 30 22]
Max over the rows:               [12 10 11  9]
Max *position* over the columns: [0 1 2]


### <span style="color:red"> Broadcasting a row onto a table </span>

<font size = "4">


Earlier, `small + 10` added 10 to every element. We can also combine a table with a single **row** of values: NumPy **broadcasts** (stretches) the row so it lines up with every row of the table.

In [ ]:
tiny = np.array([ [1,  2,  3,  4],
                  [5,  6,  7,  8],
                  [9, 10, 11, 12] ])

multipliers = np.array([1, 10, 100, 1000])
display(tiny)
display(multipliers)

print("Shape of tiny:       ", tiny.shape)
print("Shape of multipliers:", multipliers.shape)
print()
print("tiny * multipliers:")
display(tiny * multipliers)

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

array([   1,   10,  100, 1000])

Shape of tiny:        (3, 4)
Shape of multipliers: (4,)

tiny * multipliers:


array([[    1,    20,   300,  4000],
       [    5,    60,   700,  8000],
       [    9,   100,  1100, 12000]])

<font size = "4">


**The broadcasting rule:** line up the two shapes **from the right**. Each pair of sizes must be **equal**, or one of them must be **1** (or missing). A size of 1 or a missing size gets stretched to match.
```text
tiny:         (3, 4)
multipliers:     (4,)    ← 4 matches 4, and the missing size is stretched to 3
result:       (3, 4)
```

### <span style="color:red"> Application: a gradebook </span>

<font size = "4">


The array `scores` has one row for each of 100,000 students and one column for each of 7 assignments: 5 homeworks, a midterm, and a final. The array `weights` says how much each assignment counts toward the course grade.

In [ ]:
np.random.seed(22)
n_students = 100_000
assignments = ["HW1", "HW2", "HW3", "HW4", "HW5", "Midterm", "Final"]

scores = np.random.randint(low=50, high=101, size=(n_students, len(assignments)))
weights = np.array([0.06, 0.06, 0.06, 0.06, 0.06, 0.30, 0.40])

print(scores.shape, weights.shape)
print(weights.sum())
print(scores[:3])

(100000, 7) (7,)
1.0
[[54 94 50 86 88 93 70]
 [58 81 86 79 68 64 95]
 [74 79 89 84 91 58 90]]


<font size = "4">


**Weighted average for each student.** 

- Using loops, we need one loop over the students and another over the assignments.

- Using NumPy vectorization, multiply the two arrays (using broadcasting). Then sum along the columns

In [ ]:
def weighted_avg_loop(scores, weights):
    n_rows, n_cols = scores.shape
    result = []
    for i in range(n_rows):
        total = 0
        for j in range(n_cols):
            total = total + scores[i, j] * weights[j]
        result.append(total)
    return result


def weighted_avg_vec(scores, weights):
    return (scores * weights).sum(axis = 1)

n_runs = 5

time_avg_loop = timeit(lambda: weighted_avg_loop(scores, weights), number = n_runs) / n_runs
time_avg_vec = timeit(lambda: weighted_avg_vec(scores, weights), number = n_runs) / n_runs

print("nested loops:", time_avg_loop)
print("vectorized:  ", time_avg_vec)
print("speedup:     ", time_avg_loop / time_avg_vec)
print("Same answer? ", np.allclose(weighted_avg_loop(scores, weights), weighted_avg_vec(scores, weights)))


nested loops: 0.9782617600169032
vectorized:   0.00992626000661403
speedup:      98.55290505840784
Same answer?  True


<font size = "4">

`(scores * weights).sum(axis = 1)` does two vectorized steps:

1. `scores * weights` broadcasts `weights`, shape `(7,)`, onto every row of `scores`, shape `(100000, 7)`
2. `.sum(axis = 1)` adds across each row, giving one weighted average per student, shape `(100000,)`

(If you have seen linear algebra: this is a matrix-vector product, which NumPy can also write as `scores @ weights`.)

`np.allclose` checks whether two arrays are equal **up to tiny rounding errors**, which can happen when decimals are added in a different order.

<font size = "4">

**Standardizing each assignment.** To compare assignments that have different averages, we can turn each score into a **z-score**: subtract that assignment's mean and divide by its standard deviation. The means and standard deviations each have one value per column, shape `(7,)`, so they broadcast onto every row:

In [ ]:
column_means = scores.mean(axis = 0)
column_stds = scores.std(axis = 0)
z_scores = (scores - column_means) / column_stds

print(column_means.shape, z_scores.shape)
print("Every column has mean 0?", np.allclose(z_scores.mean(axis = 0), 0))
print("Every column has std 1? ", np.allclose(z_scores.std(axis = 0), 1))

(7,) (100000, 7)
Every column has mean 0? True
Every column has std 1?  True


<font size = "4">

**When the shapes don't line up.** What if we want to subtract each **student's** own average (one value per row)? `scores.mean(axis = 1)` has shape `(100000,)`. Lined up from the right, 100000 doesn't match 7, so NumPy raises an error:

In [ ]:
row_means = scores.mean(axis = 1)
print(scores.shape, row_means.shape)

scores - row_means

(100000, 7) (100000,)


ValueError: operands could not be broadcast together with shapes (100000,7) (100000,) 

<font size = "4">


`keepdims = True` keeps the collapsed axis as a size of 1, so the shape becomes `(100000, 1)`. Now the 1 is stretched across the 7 columns:

```text
scores:     (100000, 7)
row_means:  (100000, 1)    ← 1 gets stretched to 7
result:     (100000, 7)
```

In [ ]:
row_means = scores.mean(axis = 1, keepdims = True)
print(scores.shape, row_means.shape)

relative = scores - row_means
print(relative[:3].round(1))

(100000, 7) (100000, 1)
[[-22.4  17.6 -26.4   9.6  11.6  16.6  -6.4]
 [-17.9   5.1  10.1   3.1  -7.9 -11.9  19.1]
 [ -6.7  -1.7   8.3   3.3  10.3 -22.7   9.3]]


<font size = "4">


**Exercise # 4 (axes):** Using `scores` and `weights` from above, and **without any loops**:

- Store the highest score on **each assignment** in `best_per_assignment` (7 values).
- Store the **row number** of the student with the highest weighted average in `top_student`. Hint: `weighted_avg_vec` and `.argmax()`.
- The first 5 columns are homeworks: `scores[:, :5]`. Each student's **lowest homework is dropped**. Store each student's average of their other 4 homeworks in `hw_avg_drop_lowest` (100,000 values). Hint: total minus lowest, divided by 4.

In [ ]:
# Your code here
scores_small = scores[:5]
print(scores_small)
print()
print(weights)
print()
best_per_assignment = scores_small.max(axis = 0)
print(scores_small * weights)
print()
top_student = (scores_small * weights).sum(axis=1).argmax()
top_student = 

homework = scores_small[:, :5]
print(homework)
(homework.sum(axis=1)-homework.min(axis=1))/4

[[54 94 50 86 88 93 70]
 [58 81 86 79 68 64 95]
 [74 79 89 84 91 58 90]
 [55 97 77 70 84 79 68]
 [67 88 69 69 52 77 57]]

[0.06 0.06 0.06 0.06 0.06 0.3  0.4 ]

[[ 3.24  5.64  3.    5.16  5.28 27.9  28.  ]
 [ 3.48  4.86  5.16  4.74  4.08 19.2  38.  ]
 [ 4.44  4.74  5.34  5.04  5.46 17.4  36.  ]
 [ 3.3   5.82  4.62  4.2   5.04 23.7  27.2 ]
 [ 4.02  5.28  4.14  4.14  3.12 23.1  22.8 ]]

[[54 94 50 86 88]
 [58 81 86 79 68]
 [74 79 89 84 91]
 [55 97 77 70 84]
 [67 88 69 69 52]]


array([80.5 , 78.5 , 85.75, 82.  , 73.25])

# <span style="color:red"> IV. Vectorization with pandas </span>

<font size = "4">

Each column of a DataFrame (a **pandas Series**) stores its data in an array, so arithmetic on whole columns runs efficiently just like NumPy.

We'll start with an example. Our data is a (made-up) dataset of 100,000 sales **leads** - people who might become customers.

**Note:** By default, `pd.read_csv` treats certain text values, including `"None"` and `"NA"`, as missing values. `keep_default_na = False` keeps them as ordinary text. We'll use the text `"None"` in Section V.

In [ ]:
df = pd.read_csv("data/leads_100_000.csv", keep_default_na = False)
# df = pd.read_csv("data/leads_4million.csv", keep_default_na = False)  # n_runs = 1 takes ~35.3 s

print(df.shape)
df.head()

(100000, 8)


,lead_id,name,email,lead_source,lead_date,status_at_time_of_lead,current_status,estimated_value
0,L000001,Emma Patel,emma.patel@example.com,LinkedIn,2026-05-25,Contacted,None,5000
1,L000002,Jack Nguyen,jack.nguyen@example.com,Referral,2026-08-09,New,None,10000
2,L000003,Ruby Davis,ruby.davis@example.com,Email Campaign,2026-01-18,Contacted,Closed Lost,25000
3,L000004,Lucas Lopez,lucas.lopez@example.com,LinkedIn,2026-05-27,New,Contacted,25000
4,L000005,Isla Johnson,isla.johnson@example.com,Website,2026-03-25,Contacted,Proposal Sent,5000


<font size = "4">

**Example:** a salesperson earns a commission of 10% of each lead's `estimated_value`. Here are four ways to compute this new column, from slowest to fastest:

1. A `for` loop over `df.iterrows()`
2. `df.apply(func, axis = 1)`: calls `func` once for each **row**
3. `df["col"].apply(func)`: calls `func` once for each **value** in one column
4. **Vectorized** column operations: `df["col"] * 0.10`

What is `df.iterrows()` ?


In [ ]:
df_small = df.iloc[:3]
display(df_small)

for index, row in df_small.iterrows():
    print("index:", index)
    print("row:")
    display(row)
    print()
    

,lead_id,name,email,lead_source,lead_date,status_at_time_of_lead,current_status,estimated_value
0,L000001,Emma Patel,emma.patel@example.com,LinkedIn,2026-05-25,Contacted,None,5000
1,L000002,Jack Nguyen,jack.nguyen@example.com,Referral,2026-08-09,New,None,10000
2,L000003,Ruby Davis,ruby.davis@example.com,Email Campaign,2026-01-18,Contacted,Closed Lost,25000


index: 0
row:


lead_id                                  L000001
name                                  Emma Patel
email                     emma.patel@example.com
lead_source                             LinkedIn
lead_date                             2026-05-25
status_at_time_of_lead                 Contacted
current_status                              None
estimated_value                             5000
Name: 0, dtype: object


index: 1
row:


lead_id                                   L000002
name                                  Jack Nguyen
email                     jack.nguyen@example.com
lead_source                              Referral
lead_date                              2026-08-09
status_at_time_of_lead                        New
current_status                               None
estimated_value                             10000
Name: 1, dtype: object


index: 2
row:


lead_id                                  L000003
name                                  Ruby Davis
email                     ruby.davis@example.com
lead_source                       Email Campaign
lead_date                             2026-01-18
status_at_time_of_lead                 Contacted
current_status                       Closed Lost
estimated_value                            25000
Name: 2, dtype: object

<font size = "4">

Now, let's compute the commission in 4 ways:

In [ ]:
# Method 1:
def commission_iterrows(df):
    result = []
    for index, row in df.iterrows():
        result.append(row["estimated_value"] * 0.10)
    return result

method1 = lambda: commission_iterrows(df)

# Method #2
def commission_row(row):
    return row["estimated_value"] * .10

method2 = lambda: df.apply(commission_row, axis = 1)    # apply the function to each row (along axis 1)

# Method #3
def multiply_val(val):
    return val * 0.1

method3 = lambda: df["estimated_value"].apply(multiply_val)


# Method #4
method4 = lambda: df["estimated_value"] * 0.1

n_runs = 1

time_iterrows = timeit(method1, number = n_runs) / n_runs
time_apply_rows = timeit(method2, number = n_runs) / n_runs
time_apply_col = timeit(method3, number = n_runs) / n_runs
time_vectorized = timeit(method4, number = n_runs) / n_runs

times = pd.Series({"iterrows":             time_iterrows,
                   "apply(axis = 1)":      time_apply_rows,
                   "column .apply":        time_apply_col,
                   "vectorized":           time_vectorized})
times.name("average_time")
display(times)
speed_up = time_iterrows / times

timing_results = pd.DataFrame({"Avg. time (seconds)" : times, "Speedup vs. df.iterrows" : speed_up})

timing_results

TypeError: 'NoneType' object is not callable

<font size = "4">

**Exercise #5 (from Quiz 4)**

In a chemistry lab, students prepared samples of water to be brought to a boil. For each sample, they recorded the mass of the water and its starting temperature, using a thermometer that reads in degrees Fahrenheit.

Masses were recorded in **grams (g)**, and temperatures were recorded in **degrees Fahrenheit (°F)**.

1. Read in the file “water_heating.csv” from the quiz_data folder and save it as a DataFrame.

2. Using the `.apply` method, define a Pandas Series that contains the starting temperature in degrees Celsius, using this formula:

$$ \textrm{temperature in °C} = \frac{5}{9} \cdot (\textrm{temperature in °F} - 32)$$

3. Add a new column to the DataFrame named “Heat to Boil (J)” that contains the heat energy needed to bring each sample to its boiling point of 100 °C. The formula for heat energy is given below, where $m$ is the mass in grams, $c = 4.184$ is the specific heat of water (in units of J/(g·°C)), and $T$ is the starting temperature **in degrees Celsius**.

$$ \textrm{heat energy} = m \cdot c \cdot (100 - T)$$


**Here is the solution I had in mind:**

In [ ]:
df_water = pd.read_csv("data/water_heating.csv")

def convert_temp(F):
    return 5 * (F - 32) / 9

temp_C = df_water["Temp (F)"].apply(convert_temp)

df_water["Heat to Boil (J)"] = df_water["Mass (g)"] * 4.184 * (100 - temp_C)

df_water.head()

,Sample ID,Mass (g),Temp (F),Heat to Boil (J)
0,W001,100.43,61.0,35250.037289
1,W002,203.86,39.6,81693.678542
2,W003,198.33,71.5,64771.492867
3,W004,147.78,120.1,31568.238160
4,W005,147.74,69.1,49073.778036


<font size = "4">

Rewrite the code to compute the additional column "Heat to Boil (J)" using 4 different approaches:

1. Loop over `df_water.iterrows()`, appending each computed heat value to a **list**

2. Write a function that takes in a single row of a DataFrame as input, and then use `.apply` to apply it to each row of "df_water". This should result in a single Pandas Series.

3. Write a function that takes in a single temperature **value** in Fahrenheit and returns the corresponding Celsius value. Use `.apply` along with your function on the "Temp (F)" column, and assign the output to a new column of the DataFrame named "Temp (C)". Then write a function that takes in a single row of a DataFrame as input and returns the corresponding heat value. Then use `.apply` to apply it to each row of "df_water" (which now has a new column called "Temp (C)"). This should result in a single Pandas Series.

4. Use only element-wise operations on the columns of `df_water`. This should result in a single Pandas Series.

Then time each method over 10 runs, printing the average time for each, along with the speed-up with respect to looping over `df_water.iterrows()`.

In [ ]:
# Your code here
# Method 1
df_water = pd.read_csv("data/water_heating.csv")

def heat_iterrows(df):
    result = []
    for index, row in df.iterrows():
        temp_c = 5 * (row["Temp (F)"] - 32) / 9
        result.append(row["Mass (g)"] * 41.84 * (100 - temp_c))
    return result
method1 = lambda: heat_iterrows(df_water)

# Method 2
def heat_row(row):
    temp_c = 5 * (row["Temp (F)"] - 32) / 9
    return row["Mass (g)"] * 4.184 * (100-temp_c)

method2 = lambda: df_water.apply(heat_row, axis = 1)

# Method 3
def convert_to_celcius(val):
    return 5 * (val - 32) / 9

def add_celcius_column(df):
    df["Temp (C)"] = df["Temp in (F)"].apply(convert_to_celcius)

def heat_row2(row):
    return row["Mass (g)"] * 4.184 * (100-row["Temp (C)"])

def method3():
    add_celcius_column(df_water)
    return df_water.apply(heat_row2, axis=1)

#method 4
def method4():
    temp_c = 5 * (df_water["Temp (F)"] - 32) / 9
    return df_water["Mass (g)"] * 4.184 * (100-temp_c)

n_runs = 10
time_iterrows = timeit(method1, number = n_runs) / n_runs
time_iterrows = timeit(method2, number = n_runs) / n_runs
time_iterrows = timeit(method3, number = n_runs) / n_runs
time_iterrows = timeit(method4, number = n_runs) / n_runs

times = pd.Series({"iterrows:", time_iterrows, "apply(axis=1)", time_apply_rows,
                    "apply (col)"})

,Sample ID,Mass (g),Temp (F)
0,W001,100.43,61.0
1,W002,203.86,39.6
2,W003,198.33,71.5
3,W004,147.78,120.1
4,W005,147.74,69.1
5,W006,246.37,73.0
6,W007,391.41,142.4
7,W008,98.63,71.1
8,W009,254.85,104.4
9,W010,99.11,104.4


# <span style="color:red"> V. Vectorizing Conditionals </span>

<font size = "4">

Most functions that people apply to each row contain an `if` statement. These are the hardest to vectorize, and where we can gain the most speed.

**Example:** each lead has a `status_at_time_of_lead` and a `current_status`. If the lead hasn't been updated yet, `current_status` is the text `"None"`. We want a column `"status"` with the most up-to-date status:

- If `current_status` is `"None"`, use `status_at_time_of_lead`
- Otherwise, use `current_status`

In [ ]:
df = pd.read_csv("data/leads_100_000.csv", keep_default_na = False)

def set_lead_status(row):
    if row["current_status"] == "None":
        return row["status_at_time_of_lead"]
    else:
        return row["current_status"]

n_runs = 5

time_status_apply = timeit(lambda: df.apply(set_lead_status, axis = 1), number = n_runs) / n_runs
df["status"] = df.apply(set_lead_status, axis = 1)

print("apply(axis = 1):", time_status_apply)
df[["status_at_time_of_lead", "current_status", "status"]].head()

apply(axis = 1): 2.1705738000106067


,status_at_time_of_lead,current_status,status
0,Contacted,None,Contacted
1,New,None,New
2,Contacted,Closed Lost,Closed Lost
3,New,Contacted,Contacted
4,Contacted,Proposal Sent,Proposal Sent


<font size = "4">


**First attempt:** what if we pass whole columns to a function with the same `if`?

In [ ]:
def set_lead_status_broken(current, at_time):
    if current == "None":
        return at_time
    else:
        return current

set_lead_status_broken(df["current_status"], df["status_at_time_of_lead"])

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

<font size = "4">


An `if` statement needs **one** `True` or `False`. But `df["current_status"] == "None"` is a Series of **100,000** `True`/`False` values. Should the `if` run if *any* of them are `True`? *All* of them? Python refuses to guess, so it raises an error: "the truth value of a Series is ambiguous".

### <span style="color:red"> np.where: a vectorized if/else </span>

<font size = "4">


```python
np.where(condition, value_if_true, value_if_false)
```

- `condition` is a Boolean mask
- `value_if_true` and `value_if_false` can be single values, or arrays/columns that are matched up with the condition element by element
- The result is a NumPy array

In [ ]:
values = np.array([3, 8, 1, 9, 4, 7])
out = np.where(values > 5, "big", "small")
print(out)

<font size = "4">

We can use `np.where` with Pandas Series as well. The series can be the "value_if_true" or "value_if_false" too! 

In [ ]:
# Create shorter series for convenience

series1 = df["current_status"].iloc[:5]
series2 = df["status_at_time_of_lead"].iloc[:5]

display(series1) 
display(series2)

out = np.where(series1 == "None", series2, series1)

display(out)

0             None
1             None
2      Closed Lost
3        Contacted
4    Proposal Sent
Name: current_status, dtype: str

0    Contacted
1          New
2    Contacted
3          New
4    Contacted
Name: status_at_time_of_lead, dtype: str

array(['Contacted', 'New', 'Closed Lost', 'Contacted', 'Proposal Sent'],
      dtype=object)

<font size = "4">

Let's time the vectorized version, verifying that it computes the same things as the non-vectorized version:

In [ ]:
def set_lead_status_vec(current, at_time):
    return np.where(current == "None", at_time, current)

method = lambda: set_lead_status_vec(df["current_status"], df["status_at_time_of_lead"])

n_runs = 10
time_status_where = timeit(method, number = n_runs) / n_runs

df["status_2"] = set_lead_status_vec(df["current_status"], df["status_at_time_of_lead"])

print("apply(axis = 1):", time_status_apply)
print("np.where:       ", time_status_where)
print("speedup:        ", time_status_apply / time_status_where)
print("Same answer?    ", (df["status"] == df["status_2"]).all())

apply(axis = 1): 2.1705738000106067
np.where:        0.05794336000690237
speedup:         37.460268092013344
Same answer?     True


### <span style="color:red"> Combining conditions: & | ~ </span>

<font size = "4">


Python's `and`, `or`, and `not` need a single `True`/`False`, so they fail on a Series for the same reason the `if` did. Use the element-wise versions instead:

| One value | Element by element |
| --- | --- |
| `and` | `&` |
| `or` | `\|` |
| `not` | `~` |

**Always wrap each comparison in parentheses.** Python runs `&` *before* `==`, so

```python
df["lead_source"] == "LinkedIn" & df["estimated_value"] >= 50000
```

is read as `df["lead_source"] == ("LinkedIn" & df["estimated_value"]) >= 50000`, which is an error.

**Example:** How many LinkedIn leads are worth at least \$50,000 and are still open (not Closed Won or Closed Lost)?

In [ ]:
open_big_linkedin = ((df["lead_source"] == "LinkedIn")
                     & (df["estimated_value"] >= 50000)
                     & (~df["status"].isin(["Closed Won", "Closed Lost"])))

print(open_big_linkedin.sum())
display(open_big_linkedin)

2147


0        False
1        False
2        False
3        False
4        False
         ...  
99995    False
99996    False
99997    False
99998    False
99999    False
Length: 100000, dtype: bool



<font size = "4">

The Series `open_big_linkedin` contains all Boolean values. We can use it as a mask using either `.loc` or `.iloc`

In [ ]:
display(df.loc[open_big_linkedin])
display(df.iloc[open_big_linkedin])

,lead_id,name,email,lead_source,lead_date,status_at_time_of_lead,current_status,estimated_value,status,status_2
64,L000065,Mason Chen,mason.chen@example.com,LinkedIn,2026-03-11,Contacted,Negotiation,75000,Negotiation,Negotiation
67,L000068,Ruby Lopez,ruby.lopez@example.com,LinkedIn,2026-04-01,New,None,75000,New,New
141,L000142,Elijah Lewis,elijah.lewis@example.com,LinkedIn,2026-10-09,New,Contacted,75000,Contacted,Contacted
158,L000159,Aria Chen,aria.chen@example.com,LinkedIn,2026-03-09,Proposal Sent,None,75000,Proposal Sent,Proposal Sent
173,L000174,Ruby Davis,ruby.davis@example.com,LinkedIn,2026-03-01,Proposal Sent,Proposal Sent,75000,Proposal Sent,Proposal Sent
...,...,...,...,...,...,...,...,...,...,...
99662,L099663,Leah Lewis,leah.lewis@example.com,LinkedIn,2026-09-09,Contacted,Proposal Sent,75000,Proposal Sent,Proposal Sent
99805,L099806,Isla Martinez,isla.martinez@example.com,LinkedIn,2026-02-17,New,Qualified,75000,Qualified,Qualified
99935,L099936,Mason Wilson,mason.wilson@example.com,LinkedIn,2026-08-20,New,Contacted,75000,Contacted,Contacted
99957,L099958,Ethan Davis,ethan.davis@example.com,LinkedIn,2026-10-28,Proposal Sent,None,75000,Proposal Sent,Proposal Sent


,lead_id,name,email,lead_source,lead_date,status_at_time_of_lead,current_status,estimated_value,status,status_2
64,L000065,Mason Chen,mason.chen@example.com,LinkedIn,2026-03-11,Contacted,Negotiation,75000,Negotiation,Negotiation
67,L000068,Ruby Lopez,ruby.lopez@example.com,LinkedIn,2026-04-01,New,None,75000,New,New
141,L000142,Elijah Lewis,elijah.lewis@example.com,LinkedIn,2026-10-09,New,Contacted,75000,Contacted,Contacted
158,L000159,Aria Chen,aria.chen@example.com,LinkedIn,2026-03-09,Proposal Sent,None,75000,Proposal Sent,Proposal Sent
173,L000174,Ruby Davis,ruby.davis@example.com,LinkedIn,2026-03-01,Proposal Sent,Proposal Sent,75000,Proposal Sent,Proposal Sent
...,...,...,...,...,...,...,...,...,...,...
99662,L099663,Leah Lewis,leah.lewis@example.com,LinkedIn,2026-09-09,Contacted,Proposal Sent,75000,Proposal Sent,Proposal Sent
99805,L099806,Isla Martinez,isla.martinez@example.com,LinkedIn,2026-02-17,New,Qualified,75000,Qualified,Qualified
99935,L099936,Mason Wilson,mason.wilson@example.com,LinkedIn,2026-08-20,New,Contacted,75000,Contacted,Contacted
99957,L099958,Ethan Davis,ethan.davis@example.com,LinkedIn,2026-10-28,Proposal Sent,None,75000,Proposal Sent,Proposal Sent


<font size = "4">

It looks the `.query` method already utilizes vectorization. I think it's slightly harder to read than using masking though.

In [ ]:
def method1():
    open_big_linkedin = ((df["lead_source"] == "LinkedIn")
                     & (df["estimated_value"] >= 50000)
                     & (~df["status"].isin(["Closed Won", "Closed Lost"])))
    return df.loc[open_big_linkedin]

def method2():
    expr = "(lead_source == 'LinkedIn') and (estimated_value >= 50000) and (status not in ['Closed Won', 'Closed Lost'])"
    return df.query(expr) 



n_runs = 10
time_status_vector = timeit(method1, number = n_runs) / n_runs
time_status_query = timeit(method2, number = n_runs) / n_runs


print("vectorized:   ", time_status_vector)
print(".query:       ", time_status_query)

vectorized:    0.02086408999748528
.query:        0.022392770007718353


### <span style="color:red"> Multiple conditions: np.select </span>

<font size = "4">


What about `if` / `elif` / `elif` / `else`? The file "lead_categories.csv" has a status code for each lead, and we want to put each lead into a category:

In [9]:
leads = pd.read_csv("data/lead_categories.csv")
leads.head()

,lead_id,date_col1,date_col2,norm_status,lead_source,region
0,LD000001,2026-03-31,2026-05-12,EM-OPEN,Trade Show,Southeast
1,LD000002,2025-10-19,2025-10-19,DUPLICATE,Trade Show,Midwest
2,LD000003,2025-11-25,2026-03-15,EM-WEBINAR,Cold Call,Northeast
3,LD000004,2026-05-19,2026-09-05,MT-MEETING,LinkedIn,Northeast
4,LD000005,2025-10-04,2026-01-25,MT-CALL,LinkedIn,West


In [10]:
email_touches = ["EM-OPEN", "EM-CLICK", "EM-REPLY", "EM-WEBINAR"]
multi_touches = ["MT-CALL", "MT-DEMO", "MT-MEETING", "MT-PROPOSAL"]


def lead_category(row):
    if row["date_col1"] == row["date_col2"]:
        return "New Lead"
    elif row["norm_status"].startswith("CLI"):
        return "Client Lead"
    elif row["norm_status"] in multi_touches:
        return "MTouch Lead"
    elif row["norm_status"] in email_touches:
        return "EMTouch Lead"
    else:
        return "NA"

# time it (it's kind of long)
n_runs = 1
time_category_apply = timeit(lambda: leads.apply(lead_category, axis = 1), number = n_runs) / n_runs

# add the column to the DataFrame
leads["category_apply"] = leads.apply(lead_category, axis = 1)

print("apply(axis = 1):", time_category_apply)

apply(axis = 1): 88.64570660004392


<font size = "4">


**Approach 1: nested `np.where`.** Each `else` becomes another `np.where`. It works, but it's hard to read and easy to get the parentheses wrong.

Note the vectorized versions of the conditions: `.str.startswith()` instead of `.startswith()`, and `.isin(list)` instead of `in list`.

In [11]:
leads["category_where"] = \
    np.where(leads["date_col1"] == leads["date_col2"], "New Lead",
        np.where(leads["norm_status"].str.startswith("CLI"), "Client Lead",
            np.where(leads["norm_status"].isin(multi_touches), "MTouch Lead",
                np.where(leads["norm_status"].isin(email_touches), "EMTouch Lead",
                         "NA"))))

leads.head()

def method():
    return np.where(leads["date_col1"] == leads["date_col2"], "New Lead",
            np.where(leads["norm_status"].str.startswith("CLI"), "Client Lead",
                np.where(leads["norm_status"].isin(multi_touches), "MTouch Lead",
                    np.where(leads["norm_status"].isin(email_touches), "EMTouch Lead",
                             "NA"))))

n_runs = 10
time_category_nested = timeit(method, number = n_runs) / n_runs
print(time_category_nested)

0.08703848998993635


<font size = "4">


**Approach 2: `np.select`.**

```python
np.select(conditions, choices, default = ...)
```

- `conditions` is a list of Boolean masks, and `choices` is a list of values, one for each condition.
- For each row, `np.select` uses the choice for the **first** condition that is `True`, just like `if`/`elif` stops at the first `True` condition.
- Rows where no condition is `True` get the `default`, like `else`.

In [12]:
def category_select(leads):
    conditions = [
        leads["date_col1"] == leads["date_col2"],
        leads["norm_status"].str.startswith("CLI"),
        leads["norm_status"].isin(multi_touches),
        leads["norm_status"].isin(email_touches),
    ]
    choices = [
        "New Lead",
        "Client Lead",
        "MTouch Lead",
        "EMTouch Lead",
    ]
    return np.select(conditions, choices, default = "NA")

n_runs = 10
time_category_select = timeit(lambda: category_select(leads), number = n_runs) / n_runs
leads["category_select"] = category_select(leads)

print("apply(axis = 1):", time_category_apply)
print("nested np.where:", time_category_nested)
print("np.select:      ", time_category_select)
print("speedup where:  ", time_category_apply / time_category_nested)
print("speedup select: ", time_category_apply / time_category_select)
print("Same answer?    ", (leads["category_apply"] == leads["category_select"]).all())
print("Same answer?    ", (leads["category_where"] == leads["category_select"]).all())

apply(axis = 1): 88.64570660004392
nested np.where: 0.08703848998993635
np.select:       0.059291130001656714
speedup where:   1018.4655847119292
speedup select:  1495.0922102103127
Same answer?     True
Same answer?     True


<font size = "4">

**Order matters!** If we checked `email_touches` *before* the dates, a lead whose dates match **and** whose status is "EM-OPEN" would become "EMTouch Lead" instead of "New Lead":

In [ ]:
wrong_order = np.select(
    [leads["norm_status"].isin(email_touches), leads["date_col1"] == leads["date_col2"]],
    ["EMTouch Lead", "New Lead"],
    default = "other")

print("New Leads (correct order):", (leads["category_select"] == "New Lead").sum())
print("New Leads (wrong order):  ", (wrong_order == "New Lead").sum())

<font size = "4">


**Exercise 6 (nested conditions with `np.select`):** The file "sub_conditional.csv" has one row per customer account. The function `sub_conditional` below uses **nested** `if` statements to give each account a type. The cell also runs it with `apply` and times it.

Without `apply`, create a column `"lead_type"` with the same values, using `np.select`:

- `np.select` needs a **flat** list of conditions, so an `if` inside another `if` has to become one condition joined with `&`. For example, "active, with 0 providers" is `(accounts["Inactive"] == "No") & (accounts["Providers"] == 0)`.
- Because `np.select` uses the **first** `True` condition, a later condition can count on the earlier ones being `False`. That keeps the later conditions short.
- `duplicate_leads` and `bad_leads` are already `True`/`False` columns, so they can be used as masks directly.

Time your version with `number = 10`, store the average in `time_sub_select`, and print the speedup over `time_sub_apply`.

In [13]:
accounts = pd.read_csv("data/sub_conditional.csv")


def sub_conditional(row):
    if row["Inactive"] == "No":
        if row["Providers"] == 0:
            return "active_no_providers"
        elif row["Providers"] < 5:
            return "active_small"
        else:
            return "active_normal"
    elif row["duplicate_leads"]:
        return "is_dup"
    else:
        if row["bad_leads"]:
            return "inactive_bad"
        else:
            return "inactive_good"


n_runs = 1
time_sub_apply = timeit(lambda: accounts.apply(sub_conditional, axis = 1), number = n_runs) / n_runs
accounts["lead_type_apply"] = accounts.apply(sub_conditional, axis = 1)

print("apply(axis = 1):", time_sub_apply)
accounts.head()

apply(axis = 1): 6.851896699983627


,account_id,account_name,Inactive,Providers,duplicate_leads,bad_leads,region,lead_type_apply
0,AC000001,Account 1,No,0,False,False,Northeast,active_no_providers
1,AC000002,Account 2,No,3,False,True,Northeast,active_small
2,AC000003,Account 3,Yes,22,False,True,Midwest,inactive_bad
3,AC000004,Account 4,No,38,False,False,West,active_normal
4,AC000005,Account 5,No,39,True,False,Southeast,active_normal


In [ ]:
# your code here

In [ ]:
def sub_select(accounts):
    active = (row["Inactive"] == "No")
    